# Full-Spectrum Splitting-Function PLI

Phase 1 — spectral inventory.

In [ ]:
import sys
import os

# Ensure paper_demos directory is on the path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import matplotlib.pyplot as plt

from normal_mode_kernel_utils import NormalModeDataRegistry, NormalModeKernelCatalog
from full_spectrum_utils import BlockIndex, enumerate_blocks, block_data_split

DATA_DIR = 'data/normal-mode-data'
KERNEL_DIR = 'data/normal-mode-kernels/kernels-all_PREM-layers_Adrian'
S_MAX = 4

catalog = NormalModeKernelCatalog(KERNEL_DIR)
reg = NormalModeDataRegistry(DATA_DIR, mode_filter=catalog.list_modes())

blocks = enumerate_blocks(reg, s_max=S_MAX)
split = block_data_split(reg, blocks)

print(f'Total observations: {len(reg)}')
print(f'Blocks found (even s \u2264 {S_MAX}): {len(blocks)}')
for b in blocks:
    print(f'  BlockIndex(s={b.s}, t={b.t}): {len(split[b])} observations')

In [ ]:
# Bar plot: data count |d_{st}| per (s, t) block, grouped by s-degree

s_values = sorted({b.s for b in blocks})
colors = plt.cm.tab10.colors
s_color = {s: colors[i % len(colors)] for i, s in enumerate(s_values)}

labels = [f'({b.s},{b.t})' for b in blocks]
counts = [len(split[b]) for b in blocks]
bar_colors = [s_color[b.s] for b in blocks]

fig, ax = plt.subplots(figsize=(max(8, len(blocks) * 0.35), 5))
x = np.arange(len(blocks))
ax.bar(x, counts, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=7)
ax.set_xlabel('$(s, t)$ block')
ax.set_ylabel('Data count $|d_{st}|$')
ax.set_title('Data count per $(s,t)$ block')

# Legend for s-degree
from matplotlib.patches import Patch
legend_handles = [Patch(color=s_color[s], label=f's = {s}') for s in s_values]
ax.legend(handles=legend_handles, title='Degree s', loc='upper right')

plt.tight_layout()
fig.savefig('data_count_per_block.png', dpi=150)
plt.show()
print('Figure saved to data_count_per_block.png')